# FREIA detector diagnostics

Use an interactive inspector to compare wavelength spectra from selected regions of the FREIA detector. This can help identify where features in a spectrum originate on the detector.

In [ ]:
%matplotlib widget
import plopp as pp
import scipp as sc

from ess import freia
from ess.freia import data
from ess.reduce.nexus.types import Filename
from ess.reduce.unwrap import TimeResolution, WavelengthDetector
from ess.reflectometry.types import SampleRun

## Load detector events

Use the simulated direct-beam run and reconstruct the wavelength of each event from its arrival time and the WFM chopper settings. To inspect another run, replace the example data with its file path.

In [ ]:
workflow = freia.FreiaMcStasWorkflow(wavelength_from='analytical')
workflow[Filename[SampleRun]] = data.freia_mcstas_sample_run()
workflow[TimeResolution] = sc.scalar(20.0, unit='us')

events = workflow.compute(WavelengthDetector[SampleRun])

## Histogram by detector position and wavelength

The inspector expects three-dimensional data. Histogram the events by `longitude` and `height` in the detector's local frame, and by reconstructed `wavelength`.

In [ ]:
wavelength_bins = sc.linspace(
    'wavelength', start=1.0, stop=12.0, num=221, unit='angstrom'
)
detector_data = events.hist(
    height=64,
    longitude=120,
    wavelength=wavelength_bins,
    dim=events.dims,
)

## Inspect detector regions

The left panel shows the detector image summed over the wavelength range selected with the slider. Activate the rectangle tool in its toolbar, then left-click to add rectangles around regions of interest. The right panel shows the wavelength spectrum summed within each rectangle. Drag a rectangle with the right mouse button, resize it by dragging its vertices, or delete it with the middle mouse button.

In [ ]:
pp.inspector(
    detector_data,
    dim='wavelength',
    mode='rectangle',
    logc=True,
    cmin=1.0,
    title='FREIA detector',
)